In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

In [2]:
odr = pd.read_csv("data/age-dependency-ratio-old.csv")
internet = pd.read_csv("data/share-of-individuals-using-the-internet.csv")
tfp = pd.read_csv("data/total-factor-productivity.csv") 
patens = pd.read_csv("data/patent-applications-per-million.csv")
schooling = pd.read_csv("data/average-years-of-schooling-among-adults.csv")

In [3]:
odr = odr.rename(columns={
    "Entity": "country",
    "Code": "code",
    "Year": "year",
    "Age dependency ratio, old (% of working-age population)": "odr"
})

internet = internet.rename(columns={
    "Entity": "country",
    "Code": "code",
    "Year": "year",
    "Share of the population using the Internet": "internet_use"
})

tfp = tfp.rename(columns={
    "Entity": "country",
    "Code": "code",
    "Year": "year",
    "Total factor productivity level": "tfp"
})

patens = patens.rename(columns={
    "Entity": "country",
    "Code": "code",
    "Year": "year",
    "Patent applications per million people": "patens"
})

schooling = schooling.rename(columns={
    "Entity": "country",
    "Code": "code",
    "Year": "year",
    "Both genders": "schooling"
})

In [9]:
start = 2001
end = 2020

lagged_odr = odr.copy()

for df in [odr, tfp, internet, patens, schooling]:
    df["year"] = pd.to_numeric(df["year"], errors="coerce")

odr = odr[(odr["year"] >= start) & (odr["year"] <= end)]
tfp = tfp[(tfp["year"] >= start) & (tfp["year"] <= end)]
internet = internet[(internet["year"] >= start) & (internet["year"] <= end)]
patens = patens[(patens["year"] >= start) & (patens["year"] <= end)]

In [5]:
panel2 = internet[["country", "code", "year", "internet_use"]].merge(
    odr[["code", "year", "odr"]],
    on=["code", "year"],
    how="inner"
)

panel2 = panel2.merge(
    tfp[["code", "year", "tfp"]],
    on=["code", "year"],
    how="inner"
)

panel2 = panel2.merge(
    patens[["code", "year", "patens"]],
    on=["code", "year"],
    how="inner"
)

panel2 = panel2.merge(
    schooling[["code", "year", "schooling"]],
    on=["code", "year"],
    how="inner"
)

panel2["log_internet_use"] = np.log(panel2["internet_use"])
panel2["log_tfp"] = np.log(panel2["tfp"])
panel2["log_patens"] = np.log(panel2["patens"])

#make a copy of the dataframe
panel_2yr = panel2.copy()

# Vælg startår for perioderne
start_year = 2002

# Lav 2-års perioder
panel_2yr["period_start"] = start_year + 2 * ((panel_2yr["year"] - start_year) // 2)
panel_2yr["period_end"] = panel_2yr["period_start"] + 1

# Variabler der skal gennemsnittes
vars_to_average = [
    "tfp",
    "odr",
    "hc",
    "log_tfp",
    "log_gdppc",
    "log_internet_use",
    "patens"
]

# Behold kun dem der faktisk findes i datasættet
vars_to_average = [v for v in vars_to_average if v in panel_2yr.columns]

# Collapse til én række pr. land og 2-års periode
panel_2yr = (
    panel_2yr
    .groupby(["code", "country", "period_start", "period_end"], as_index=False)[vars_to_average]
    .mean()
)

# Lav evt. en year-variabel til FE
panel_2yr["year"] = panel_2yr["period_start"]

In [6]:
#make a copy of the dataframe
panel_3yr = panel2.copy()

# Vælg startår for perioderne
start_year = 2002

# Lav 3-års perioder
panel_3yr["period_start"] = start_year + 3 * ((panel_3yr["year"] - start_year) // 3)
panel_3yr["period_end"] = panel_3yr["period_start"] + 2

# Behold kun dem der faktisk findes i datasættet
vars_to_average = [v for v in vars_to_average if v in panel_3yr.columns]

# Collapse til én række pr. land og 3-års periode
panel_3yr = (
    panel_3yr
    .groupby(["code", "country", "period_start", "period_end"], as_index=False)[vars_to_average]
    .mean()
)

# Lav evt. en year-variabel til FE
panel_3yr["year"] = panel_3yr["period_start"]

In [7]:
#make a copy of the dataframe
panel_6yr = panel2.copy()

# Vælg startår for perioderne
start_year = 2002

# Lav 6-års perioder
panel_6yr["period_start"] = start_year + 6 * ((panel_6yr["year"] - start_year) // 6)
panel_6yr["period_end"] = panel_6yr["period_start"] + 5

# Behold kun dem der faktisk findes i datasættet
vars_to_average = [v for v in vars_to_average if v in panel_6yr.columns]

# Collapse til én række pr. land og 6-års periode
panel_6yr = (
    panel_6yr       
    .groupby(["code", "country", "period_start", "period_end"], as_index=False)[vars_to_average]
    .mean()
)

# Lav evt. en year-variabel til FE
panel_6yr["year"] = panel_6yr["period_start"]

#remove country if data not observed in all periods
#country_counts = panel_6yr["code"].value_counts()
#countries_to_keep = country_counts[country_counts == 4].index
#panel_6yr = panel_6yr[panel_6yr["code"].isin(countries_to_keep)]

panel_6yr.head(10)

,code,country,period_start,period_end,tfp,odr,log_tfp,log_internet_use,patens,year
0,AGO,Angola,2014,2019,1.092402,5.177665,0.088306,3.418534,0.126743,2014
1,ALB,Albania,2008,2013,1.099752,16.544027,0.095085,3.850148,1.032633,2008
2,ALB,Albania,2014,2019,0.998426,19.241723,-0.002802,4.110855,4.919819,2014
3,ARG,Argentina,1996,2001,1.069000,15.598366,0.066723,2.280422,18.365534,1996
4,ARG,Argentina,2002,2007,1.105510,15.776194,0.097609,2.801918,22.629363,2002
5,ARG,Argentina,2008,2013,1.185111,16.366183,0.169522,3.785919,16.311510,2008
6,ARG,Argentina,2014,2019,1.091185,17.520732,0.086754,4.282401,12.124081,2014
7,ARG,Argentina,2020,2025,0.998764,18.258383,-0.001237,4.448685,20.578880,2020
8,ARM,Armenia,1996,2001,0.433463,14.565846,-0.835948,0.489249,43.589264,1996
9,ARM,Armenia,2002,2007,0.733333,16.032268,-0.326821,1.475265,56.053190,2002


## pause

In [8]:
print(panel_6yr[["internet_use", "odr", "tfp", "schooling"]].isna().sum())

KeyError: "['internet_use', 'schooling'] not in index"

In [ ]:
# kopi af det færdige balanced panel
df_est = panel2.copy()

# estimation
model3 = smf.ols(
    "tfp ~ odr  + log_internet_use + patens + C(code) + C(year)",
    data=df_est
)

results3 = model3.fit(
    cov_type="cluster",
    cov_kwds={"groups": df_est["code"]}
)

main_results3 = pd.DataFrame({
    "coef": results3.params[["odr","log_internet_use", "patens"]],
    "std_err": results3.bse[["odr", "log_internet_use", "patens"]],
    "p_value": results3.pvalues[["odr", "log_internet_use", "patens"]],
})

print(main_results3.round(4))

                    coef  std_err  p_value
odr               0.0125   0.0056   0.0248
log_internet_use  0.0543   0.0107   0.0000
patens            0.0001   0.0000   0.0304


In [ ]:
# kopi af det færdige balanced panel
df_est = panel2.copy()

# estimation
model3 = smf.ols(
    "tfp ~ odr  + log_internet_use + log_patens + C(code) + C(year)",
    data=df_est
)

results3 = model3.fit(
    cov_type="cluster",
    cov_kwds={"groups": df_est["code"]}
)

main_results3 = pd.DataFrame({
    "coef": results3.params[["odr","log_internet_use", "log_patens"]],
    "std_err": results3.bse[["odr", "log_internet_use", "log_patens"]],
    "p_value": results3.pvalues[["odr", "log_internet_use", "log_patens"]],
})

print(main_results3.round(4))

                    coef  std_err  p_value
odr               0.0104   0.0056   0.0624
log_internet_use  0.0530   0.0110   0.0000
log_patens       -0.0193   0.0153   0.2071


In [ ]:
# kopi af det færdige balanced panel
df_est = panel2.copy()

# lav log af BNP pr. capita
df_est["log_gdppc"] = np.log(df_est["gdppc"])

# estimation
model3 = smf.ols(
    "log_tfp ~ odr  + log_internet_use + patens + C(code) + C(year)",
    data=df_est
)

results3 = model3.fit(
    cov_type="cluster",
    cov_kwds={"groups": df_est["code"]}
)

main_results3 = pd.DataFrame({
    "coef": results3.params[["odr","log_internet_use", "patens"]],
    "std_err": results3.bse[["odr", "log_internet_use", "patens"]],
    "p_value": results3.pvalues[["odr", "log_internet_use", "patens"]],
})

print(main_results3.round(4))

                    coef  std_err  p_value
odr               0.0141   0.0050   0.0048
log_internet_use  0.0780   0.0165   0.0000
patens            0.0001   0.0001   0.0428


In [ ]:
# kopi af det færdige balanced panel
df_est = panel_3yr.copy()

# estimation
model3 = smf.ols(
    "log_tfp ~ odr  + log_internet_use + patens + C(code) + C(year)",
    data=df_est
)

results3 = model3.fit(
    cov_type="cluster",
    cov_kwds={"groups": df_est["code"]}
)

main_results3 = pd.DataFrame({
    "coef": results3.params[["odr","log_internet_use", "patens"]],
    "std_err": results3.bse[["odr", "log_internet_use", "patens"]],
    "p_value": results3.pvalues[["odr", "log_internet_use", "patens"]],
})

print(main_results3.round(4))
print(results3.nobs)
print(df_est["code"].nunique())

                    coef  std_err  p_value
odr               0.0156   0.0055   0.0044
log_internet_use  0.0854   0.0187   0.0000
patens            0.0001   0.0001   0.0506
698.0
106


In [ ]:
# kopi af det færdige balanced panel
df_est = panel_6yr.copy()

# estimation
model3 = smf.ols(
    "log_tfp ~ odr  + log_internet_use + patens + C(code) + C(year)",
    data=df_est
)

results3 = model3.fit(
    cov_type="cluster",
    cov_kwds={"groups": df_est["code"]}
)
# add number of observations

main_results3 = pd.DataFrame({
    "coef": results3.params[["odr","log_internet_use", "patens"]],
    "std_err": results3.bse[["odr", "log_internet_use", "patens"]],
    "p_value": results3.pvalues[["odr", "log_internet_use", "patens"]],
})

print(main_results3.round(4))
print(results3.nobs)
print(df_est["code"].nunique())


                    coef  std_err  p_value
odr               0.0185   0.0061   0.0023
log_internet_use  0.0942   0.0214   0.0000
patens            0.0001   0.0001   0.0516
445.0
106


In [ ]:
# kopi af det færdige balanced panel
df_est = panel2.copy()

# estimation
model3 = smf.ols(
    "log_tfp ~ odr + C(code) + C(year)",
    data=df_est
)

results3 = model3.fit(
    cov_type="cluster",
    cov_kwds={"groups": df_est["code"]}
)

main_results3 = pd.DataFrame({
    "coef": results3.params[["odr"]],
    "std_err": results3.bse[["odr"]],
    "p_value": results3.pvalues[["odr"]],
})

print(main_results3.round(4))

       coef  std_err  p_value
odr  0.0001   0.0048   0.9852


# lagged ODR

In [ ]:

lagged = 1
start = 2001 - lagged
end = 2020

lagged_odr = lagged_odr[(lagged_odr["year"] >= start) & (lagged_odr["year"] <= end)]
lagged_odr["odr_lagged"] = lagged_odr.groupby("code")["odr"].shift(lagged)
lagged_odr = lagged_odr.dropna(subset=["odr_lagged"])


panel2 = panel2.merge(
    lagged_odr[["code", "year", "odr_lagged"]],
    on=["code", "year"],
    how="inner"
)

In [ ]:
# kopi af det færdige balanced panel
df_est = panel2.copy()

# estimation
model3 = smf.ols(
    "log_tfp ~ odr_lagged + C(code) + C(year)",
    data=df_est
)

results3 = model3.fit(
    cov_type="cluster",
    cov_kwds={"groups": df_est["code"]}
)

main_results3 = pd.DataFrame({
    "coef": results3.params[["odr_lagged"]],
    "std_err": results3.bse[["odr_lagged"]],
    "p_value": results3.pvalues[["odr_lagged"]],
})

print(main_results3.round(4))

              coef  std_err  p_value
odr_lagged  0.0002   0.0048   0.9677


In [ ]:
# kopi af det færdige balanced panel
df_est = panel2.copy()

# estimation
model3 = smf.ols(
    "log_tfp ~ odr_lagged + log_internet_use + log_patens + C(code) + C(year)",
    data=df_est
)

results3 = model3.fit(
    cov_type="cluster",
    cov_kwds={"groups": df_est["code"]}
)

main_results3 = pd.DataFrame({
    "coef": results3.params[["odr_lagged", "log_internet_use", "log_patens"]],
    "std_err": results3.bse[["odr_lagged", "log_internet_use", "log_patens"]],
    "p_value": results3.pvalues[["odr_lagged", "log_internet_use", "log_patens"]],
})

print(main_results3.round(4))

                    coef  std_err  p_value
odr_lagged        0.0115   0.0054   0.0343
log_internet_use  0.0753   0.0150   0.0000
log_patens       -0.0219   0.0159   0.1680


In [ ]:
# kopi af det færdige balanced panel
df_est = panel2.copy()

# estimation
model3 = smf.ols(
    "log_tfp ~ odr_lagged + log_patens + C(code) + C(year)",
    data=df_est
)

results3 = model3.fit(
    cov_type="cluster",
    cov_kwds={"groups": df_est["code"]}
)

main_results3 = pd.DataFrame({
    "coef": results3.params[["odr_lagged", "log_patens"]],
    "std_err": results3.bse[["odr_lagged", "log_patens"]],
    "p_value": results3.pvalues[["odr_lagged", "log_patens"]],
})

print(main_results3.round(4))

              coef  std_err  p_value
odr_lagged -0.0009   0.0048   0.8569
log_patens -0.0212   0.0191   0.2663


In [ ]:
# kopi af det færdige balanced panel
df_est = panel2.copy()

# estimation
model3 = smf.ols(
    "log_tfp ~ odr + log_patens + C(code) + C(year)",
    data=df_est
)

results3 = model3.fit(
    cov_type="cluster",
    cov_kwds={"groups": df_est["code"]}
)

main_results3 = pd.DataFrame({
    "coef": results3.params[["odr", "log_patens"]],
    "std_err": results3.bse[["odr", "log_patens"]],
    "p_value": results3.pvalues[["odr", "log_patens"]],
})

print(main_results3.round(4))

              coef  std_err  p_value
odr        -0.0010   0.0048   0.8391
log_patens -0.0213   0.0191   0.2654
